In [0]:
%pip install yfinance

import yfinance as yf
import pandas as pd
from datetime import datetime, timezone

In [0]:
# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("schema", "valeriimatviiv_bronze", "2. Target Schema")
dbutils.widgets.text("volume", "market_radar_landing", "3. Landing Volume")
dbutils.widgets.text("tickers", "AAPL,NVDA,MSFT,AMZN,TSLA,QQQ", "4. Tickers")
dbutils.widgets.text("start_date", "2024-01-01", "5. Start Date (YYYY-MM-DD)")

# Retrieve widget values
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
tickers = [t.strip() for t in dbutils.widgets.get("tickers").split(",")]
start_date = dbutils.widgets.get("start_date")
end_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

landing_path = f"/Volumes/{catalog}/{schema}/{volume}/landing/nasdaq_price"
dbutils.fs.mkdirs(landing_path)


In [0]:
records = []

for symbol in tickers:
    print(f"Fetching daily price history for {symbol}...")
    df_symbol = yf.download(symbol, start=start_date, end=end_date, interval="1d", progress=False)
    
    if not df_symbol.empty:
        # Handle potential MultiIndex columns returned by yfinance
        if isinstance(df_symbol.columns, pd.MultiIndex):
            df_symbol.columns = df_symbol.columns.get_level_values(0)
            
        df_symbol = df_symbol.reset_index()
        df_symbol["Symbol"] = symbol
        records.append(df_symbol)

if records:
    combined_df = pd.concat(records, ignore_index=True)
    
    # Save combined raw CSV file to landing volume
    output_file = f"{landing_path}/nasdaq_price_history.csv"
    combined_df.to_csv(output_file, index=False)
    print(f"Successfully landed price history ({len(combined_df)} rows) to: {output_file}")
else:
    print("No price data fetched.")

In [0]:
# catalog = dbutils.widgets.get("catalog")
# schema = dbutils.widgets.get("schema")
# volume = dbutils.widgets.get("volume")
# landing_path = f"/Volumes/{catalog}/{schema}/{volume}/landing/nasdaq_price"

# print("--- Volume File Listing ---")
# display(dbutils.fs.ls(landing_path))

# print("--- Preview Landed CSV ---")
# df_preview = spark.read.option("header", "true").csv(f"{landing_path}/nasdaq_price_history.csv")
# print(f"Total Landed Price Rows: {df_preview.count()}")
# display(df_preview.limit(10))